In [ ]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utility import *
from _fractures import *
from _initial_conditions import *


# -- Qiskit imports --
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import MatrixExponential
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector

np.random.seed(0)

If you want try out your own grid parameters, uncommment the code in red and set your own parameters. Do not change the 4th line that defines `dx, dy, dz`, as these values are purely determined from other set values.

In [ ]:
# -- Grid parameters --

# -- Lower/Upper Bound, Number of Points, Separation Between Points --

xmin,ymin,zmin,xmax,ymax,zmax,Nx,Ny,Nz,dx,dy,dz = get_grid_parameters()

# manually setting if you want to try other things
'''
xmin,ymin,zmin = -1,-1,-1
xmax,ymax,zmax = 1,1,1
Nx, Ny, Nz = 8, 8, 8
dx, dy, dz = (xmax-xmin)/(Nx-1), (ymax-ymin)/(Ny-1), (zmax-zmin)/(Nz-1)
'''

# -- Staggered Grid Points for each wavefield component of seismic wave equation w = [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

N_main,N_vx,N_vy,N_vz,N_sxy,N_sxz,N_syz,N_vel,N_stress,psi_len = get_grid_size(Nx,Ny,Nz)

Only change `file_label_2`, which controls the initial condition (Ricker or Gaussian), and `file_label_5`, which controls fracture geometry.

In [ ]:
## Configure which physical scenario to run for the notebook, times will be determined later
#Decide if quantum result will be calculated (takes a long time to run, don't run with over 6x6x6 case
calc_quantum = False


#If quantum is true, calculate quantum and classical, if false just calculate classical
file_label_1 = 'classical'
if calc_quantum:
    file_label_1='quantum'

#initial conditions: ricker_IC_1 or gaussian_IC_1
file_label_2 = 'ricker_IC_1'

#Use grid size to create this label 
file_label_3 = str(Nx)+'x'+str(Ny)+'x'+str(Nz)

#Use physical domain for this label

file_label_4 = str(xmin)+'to'+str(xmax)

#Fracture geometry 
# options: two_perp_xy_yz, no_fracture, one_xy, one_yz
file_label_5 = 'two_perp_xy_yz'

In [ ]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).


# Density model and S matrix (3D with one horizontal fracture)

#Set fracture thickness
fracture_thickness = 0
#Set density and compliance matrix according to fracture geometry specified earlier
if file_label_5 == 'two_perp_xy_yz':
    rho_model, S_matrix = two_perpendicular_fractures(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'no_fracture':
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)
elif file_label_5 == 'one_xy':
    rho_model, S_matrix = one_horizontal_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
elif file_label_5 == 'one_yz':
    rho_model, S_matrix = one_vertical_fracture(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)
else:
    rho_model, S_matrix = no_fractures(Nx,Ny,Nz,dx,dy,dz)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)

H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
print("Number of grid points: ", psi_len)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)

In [ ]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

#phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
if file_label_2 == 'ricker_IC_1':
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
elif file_label_2 == 'gaussian_IC_1':
    phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
else:
    phi_0 = ricker_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)
    
# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])
# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

In [ ]:
# Time Integration (Has classical integration errors (DOP853))
def get_classical_state_vector(t):
    #phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='DOP853').y.T[-1] 
    phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='RK45').y.T[-1] 
    psi_t_classical = B_sqrt @ phi_t
    return psi_t_classical

Time ranges include `[5e-6,1e-4], [5e-5,1e-3],[.005,.1],[.05,1],[.25,5],[.5,10]`

I found that only the first time range makes sense, as the other time ranges get chaotic too quickly.

The `np.linspace` below is structured as `np.linspace(t_start, t_end, n_timesteps)`

In [ ]:
t_history = np.linspace(5e-6, 1e-4, 20)
history = [psi_0]
for t in t_history:
    history.append(get_classical_state_vector(t))
times = np.concatenate([[0.0], t_history])


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import gridspec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from classical_plots import build_zslice_frames_and_scales, extract_main_grid_fields_xyz


def plot_time_slider2(
    k_fixed,
    history,
    xmin,
    ymin,
    zmin,
    xmax,
    ymax,
    zmax,
    Nx,
    Ny,
    Nz,
    dx,
    dy,
    dz,
    times=None,
):
    """Z-slice 2D panels plus full-domain 3D velocity arrows using the same field pipeline as the 2D plots."""
    k_fixed = max(0, min(k_fixed, Nz - 1))
    n_steps = len(history)

    frames, global_max, plot_extent, z_phys = build_zslice_frames_and_scales(
        history, Nx, Ny, Nz, k_fixed, xmin, ymin, zmin, dx, dy, dz
    )

    # Same state vectors and extractor as build_zslice_frames_and_scales (main grid, (Nx,Ny,Nz)).
    vel_frames = []
    v_all_max = 0.0
    for phi in history:
        vxm, vym, vzm, *_ = extract_main_grid_fields_xyz(phi, Nx, Ny, Nz)
        vel_frames.append((vxm, vym, vzm))
        v_all_max = max(v_all_max, float(np.max(np.sqrt(vxm**2 + vym**2 + vzm**2))))
    if v_all_max == 0.0:
        v_all_max = 1e-30

    x = xmin + np.arange(Nx, dtype=float) * dx
    y = ymin + np.arange(Ny, dtype=float) * dy
    z = zmin + np.arange(Nz, dtype=float) * dz
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

    # Subsample for 3D quiver (full field is the same data as 2D; only arrow density is reduced).
    sx = max(1, int(np.ceil(Nx / 10)))
    sy = max(1, int(np.ceil(Ny / 10)))
    sz = max(1, int(np.ceil(Nz / 10)))
    sl = (slice(None, None, sx), slice(None, None, sy), slice(None, None, sz))
    Xs, Ys, Zs = X[sl], Y[sl], Z[sl]
    box = max((Nx - 1) * dx, (Ny - 1) * dy, (Nz - 1) * dz)
    arrow_len = 0.09 * box

    fig = plt.figure(figsize=(24, 17))
    gs = gridspec.GridSpec(4, 5, figure=fig, height_ratios=[1.0, 1.0, 1.0, 1.15], hspace=0.38, wspace=0.32)
    ax_flat = [fig.add_subplot(gs[r, c]) for r in range(3) for c in range(5)]
    ax3d = fig.add_subplot(gs[3, :], projection="3d")

    t0_lbl = f", t={times[0]:.3e} s" if times is not None and len(times) == n_steps else ", frame 0"
    title_text = fig.suptitle(
        f"Z-slice k={k_fixed} (z={z_phys:.3f} m){t0_lbl} — 2D panels + 3D velocity",
        fontsize=15,
        fontweight="bold",
    )

    layout = [
        ("v_x", "Velocity X", "RdBu"),
        ("v_y", "Velocity Y", "RdBu"),
        ("v_z", "Velocity Z", "RdBu"),
        ("v_mag", "Velocity Mag", "viridis"),
        ("sigma_xx", "Normal XX", "RdBu"),
        ("sigma_yy", "Normal YY", "RdBu"),
        ("sigma_zz", "Normal ZZ", "RdBu"),
        ("sigma_xy", "Shear XY", "RdBu"),
        ("sigma_xz", "Shear XZ", "RdBu"),
        ("sigma_yz", "Shear YZ", "RdBu"),
        ("sigdot_x", r"$(\sigma\cdot n)_x$", "RdBu"),
        ("sigdot_y", r"$(\sigma\cdot n)_y$", "RdBu"),
        ("sigdot_z", r"$(\sigma\cdot n)_z$", "RdBu"),
        ("sigdot_mag", r"$|\sigma\cdot n|$", "viridis"),
    ]

    images = {}
    for ax_idx, (key, title, cmap) in enumerate(layout):
        ax = ax_flat[ax_idx]
        data = frames[0][key]
        vmax = global_max[key]
        if vmax == 0:
            vmax = 1e-10
        if cmap == "RdBu":
            im = ax.imshow(data, cmap=cmap, origin="lower", extent=plot_extent, vmin=-vmax, vmax=vmax)
        else:
            im = ax.imshow(data, cmap=cmap, origin="lower", extent=plot_extent, vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        images[key] = im

    for ax_idx in range(len(layout), len(ax_flat)):
        ax_flat[ax_idx].axis("off")

    def draw_quiver(t_idx):
        ax3d.clear()
        vxm, vym, vzm = vel_frames[t_idx]
        Us, Vs, Ws = vxm[sl], vym[sl], vzm[sl]
        spd = np.sqrt(Us**2 + Vs**2 + Ws**2)
        q = ax3d.quiver(
            Xs,
            Ys,
            Zs,
            Us,
            Vs,
            Ws,
            length=arrow_len,
            normalize=True,
            cmap="viridis",
            clim=(0.0, v_all_max),
        )
        q.set_array(spd.ravel())
        ax3d.set_xlim(float(xmin), float(xmax))
        ax3d.set_ylim(float(ymin), float(ymax))
        ax3d.set_zlim(float(zmin), float(zmax))
        ax3d.set_xlabel("x [m]")
        ax3d.set_ylabel("y [m]")
        ax3d.set_zlabel("z [m]")
        ax3d.set_title(
            r"$\mathbf{v}$ (same `extract_main_grid_fields_xyz` as 2D): "
            r"color = $|\mathbf{v}|$, scale fixed over time"
        )
        # Same z-plane as the imshow row (visual tie-in).
        ax3d.plot(
            [xmin, xmax, xmax, xmin, xmin],
            [ymin, ymin, ymax, ymax, ymin],
            [z_phys, z_phys, z_phys, z_phys, z_phys],
            "r-",
            linewidth=1.2,
            alpha=0.85,
        )
        return q

    draw_quiver(0)

    def update(t):
        t_lbl = f", t={times[t]:.3e} s" if times is not None and t < len(times) else f", frame {t}"
        title_text.set_text(
            f"Z-slice k={k_fixed} (z={z_phys:.3f} m){t_lbl} — 2D panels + 3D velocity"
        )
        for key in images:
            images[key].set_data(frames[t][key])
        draw_quiver(t)
        return list(images.values())

    anim = FuncAnimation(fig, update, frames=n_steps, interval=200, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def plot_time_slider_sigma_dot_n_slice(
    plane,
    index,
    history,
    xmin,
    ymin,
    zmin,
    xmax,
    ymax,
    zmax,
    Nx,
    Ny,
    Nz,
    dx,
    dy,
    dz,
):
    """$(\\sigma\\cdot n)$ components and magnitude on an XY, XZ, or YZ slice (main-grid stresses)."""
    *_, psi_len = get_grid_size(Nx, Ny, Nz)

    if plane == "xy":
        index = max(0, min(index, Nz - 1))
        z0 = zmin + index * dz
        extent = [xmin, xmax, ymin, ymax]
        xlab, ylab = "x", "y"
        title_plane = f"XY slice z={z0:.4g} (k={index})"
    elif plane == "xz":
        index = max(0, min(index, Ny - 1))
        y0 = ymin + index * dy
        extent = [xmin, xmax, zmin, zmax]
        xlab, ylab = "x", "z"
        title_plane = f"XZ slice y={y0:.4g} (j={index})"
    elif plane == "yz":
        index = max(0, min(index, Nx - 1))
        x0 = xmin + index * dx
        extent = [ymin, ymax, zmin, zmax]
        xlab, ylab = "y", "z"
        title_plane = f"YZ slice x={x0:.4g} (i={index})"
    else:
        raise ValueError("plane must be 'xy', 'xz', or 'yz'")

    x_1d = np.linspace(xmin, xmax, Nx)
    y_1d = np.linspace(ymin, ymax, Ny)
    z_1d = np.linspace(zmin, zmax, Nz)

    n_steps = len(history)
    frames = []
    global_max = {k: 0.0 for k in ["sigdot_x", "sigdot_y", "sigdot_z", "sigdot_mag"]}

    for t in range(n_steps):
        phi_raw = history[t]
        ph = np.asarray(phi_raw, dtype=float)
        if np.iscomplexobj(phi_raw):
            ph = np.real(phi_raw)
        ph = ph[:psi_len]

        _, _, _, sxx, syy, szz, sxy, sxz, syz = phi_to_fields_main_grid(ph, Nx, Ny, Nz)

        if plane == "xy":
            k = index
            Axx = sxx[k, :, :]
            Ayy = syy[k, :, :]
            Azz = szz[k, :, :]
            Axy = sxy[k, :, :]
            Axz = sxz[k, :, :]
            Ayz = syz[k, :, :]
            X, Y = np.meshgrid(x_1d, y_1d)
            Z = np.full_like(X, z_1d[k], dtype=float)
        elif plane == "xz":
            j = index
            Axx = sxx[:, j, :]
            Ayy = syy[:, j, :]
            Azz = szz[:, j, :]
            Axy = sxy[:, j, :]
            Axz = sxz[:, j, :]
            Ayz = syz[:, j, :]
            X, Z = np.meshgrid(x_1d, z_1d)
            Y = np.full_like(X, y_1d[j], dtype=float)
        else:
            i = index
            Axx = sxx[:, :, i]
            Ayy = syy[:, :, i]
            Azz = szz[:, :, i]
            Axy = sxy[:, :, i]
            Axz = sxz[:, :, i]
            Ayz = syz[:, :, i]
            Y, Z = np.meshgrid(y_1d, z_1d)
            X = np.full_like(Y, x_1d[i], dtype=float)

        tiny = 1e-15
        Rmag = np.sqrt(X**2 + Y**2 + Z**2)
        Rmag = np.maximum(Rmag, tiny)
        n_x = X / Rmag
        n_y = Y / Rmag
        n_z = Z / Rmag
        sigdot_x = Axx * n_x + Axy * n_y + Axz * n_z
        sigdot_y = Axy * n_x + Ayy * n_y + Ayz * n_z
        sigdot_z = Axz * n_x + Ayz * n_y + Azz * n_z
        sigdot_mag = np.sqrt(sigdot_x**2 + sigdot_y**2 + sigdot_z**2)

        frame_data = {
            "sigdot_x": sigdot_x,
            "sigdot_y": sigdot_y,
            "sigdot_z": sigdot_z,
            "sigdot_mag": sigdot_mag,
        }
        frames.append(frame_data)
        for key in global_max:
            cm = float(np.max(np.abs(frame_data[key])))
            if cm > global_max[key]:
                global_max[key] = cm

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axf = axes.flatten()
    layout = [
        ("sigdot_x", r"$(\sigma\cdot n)_x$", "RdBu"),
        ("sigdot_y", r"$(\sigma\cdot n)_y$", "RdBu"),
        ("sigdot_z", r"$(\sigma\cdot n)_z$", "RdBu"),
        ("sigdot_mag", r"$|\sigma\cdot n|$", "viridis"),
    ]
    title_text = fig.suptitle(f"{title_plane} — frame 0", fontsize=14, fontweight="bold")

    images = {}
    for ax, (key, title, cmap) in zip(axf, layout):
        data = frames[0][key]
        vmax = global_max[key]
        if vmax == 0:
            vmax = 1e-10
        if cmap == "RdBu":
            im = ax.imshow(data, cmap=cmap, origin="lower", extent=extent, vmin=-vmax, vmax=vmax)
        else:
            im = ax.imshow(data, cmap=cmap, origin="lower", extent=extent, vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xlabel(xlab)
        ax.set_ylabel(ylab)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        images[key] = im

    plt.tight_layout()

    def update(frame):
        title_text.set_text(f"{title_plane} — frame {frame}")
        out = []
        for key in images:
            images[key].set_data(frames[frame][key])
            out.append(images[key])
        return out

    anim = FuncAnimation(fig, update, frames=n_steps, interval=200, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


The following block will create plots for each slice. If you want just a single slice, create a new code block, get rid of the for loop, and just indicate a single value for `k_fixed`, which should be an integer value in `[0, ..., Nz-1]`

In [ ]:
for i in range(Nz):
    html_widget = plot_time_slider2(
        k_fixed=i,
        history=history,
        xmin=xmin,
        ymin=ymin,
        zmin=zmin,
        xmax=xmax,
        ymax=ymax,
        zmax=zmax,
        Nx=Nx,
        Ny=Ny,
        Nz=Nz,
        dx=dx,
        dy=dy,
        dz=dz,
        times=times,
    )
    display(html_widget)